# Vielbein 계산 테스트 — FLRW DFT perturbation linear order

## 목표
다음 식을 linear order까지 component 단위로 계산한다:

$$
T_{p\bar q} \;=\; -\,\delta g_{\mu\nu}\, e^{\rho}{}_{p}\, \bar B_{\rho\sigma}\, \bar g^{\sigma\mu}\, \bar e^{\nu}{}_{\bar q}
\;-\; \delta B_{\mu\nu}\, e^{\rho}{}_{p}\, \bar B_{\rho\sigma}\, \bar g^{\sigma\mu}\, \bar e^{\nu}{}_{\bar q}
$$

- 모든 `δ`가 붙지 않은 양 → background 값
- `δg_{μν}`, `δB_{μν}`는 perturbation (symbolic matrix로 유지)
- 결과는 (unbarred lorentz $p$, barred lorentz $\bar q$) 2-index 텐서

## 진행 단계
1. IndexCalc으로 symbolic 구조 검증 (Einstein convention, free/contracted indices)
2. FLRW background 값 정의 (vielbein, 역계량, $\bar B$)
3. 중간 텐서 $Y^\mu{}_p = e^\rho{}_p\,\bar B_{\rho\sigma}\,\bar g^{\sigma\mu}$ 계산
4. $\bar e^\nu{}_{\bar q}$ 곱해 $Z^{\mu\nu}{}_{p\bar q}$ 구성
5. $\delta g$, $\delta B$와 축약해 최종 $T_{p\bar q}$

## Step 1. IndexCalc 구조 검증

세 개의 index space를 등록:
- spacetime: `μνλρσαβ`
- lorentz (unbarred): `pqrs`
- lorentz̄ (barred): `p̄q̄r̄s̄`

LaTeX로 식을 파싱해서 Einstein convention이 올바른지, free index가 $(p, \bar q)$인지 확인한다.

In [1]:
from indexcalc import (
    IndexSpace, IndexRegistry, parse, validate_einstein, to_latex,
)

st = IndexSpace('spacetime',   dim=4, indices='μνλρσαβ', metric='g')
lr = IndexSpace('lorentz',     dim=4, indices='pqrs',    metric='η')
lb = IndexSpace('lorentz_bar', dim=4, indices='p̄q̄r̄s̄',    metric='η̄')

reg = IndexRegistry()
for space in (st, lr, lb):
    reg.register(space)

# δg, δB는 파서에서 'δ' prefix가 tensor name으로 들어가야 하는데 LaTeX \delta가 그리스 문자로도 쓰이므로
# tensor name은 dg, dB 로 치환해서 파싱한다 (의미는 동일)
latex_expr = (
    r'- dg_{\mu\nu} e^{\rho}{}_{p} B_{\rho\sigma} g^{\sigma\mu} \bar{e}^{\nu}{}_{\bar{q}} '
    r'- dB_{\mu\nu} e^{\rho}{}_{p} B_{\rho\sigma} g^{\sigma\mu} \bar{e}^{\nu}{}_{\bar{q}}'
)

expr = parse(latex_expr, reg)
info = validate_einstein(expr)

print('Einstein 유효:', info['valid'])
print('Free indices :', [str(i) for i in info['free']])
print('Contracted   :', [(a.name, b.name) for a, b in info['contracted']])
print('LaTeX 재출력  :')
print(' ', to_latex(expr))

Einstein 유효: True
Free indices : ['_p', '_q̄']
Contracted   : [('μ', 'μ'), ('ν', 'ν'), ('ρ', 'ρ'), ('σ', 'σ')]
LaTeX 재출력  :
  -dg_{\mu \nu} e^{\rho}{}_{p} B_{\rho \sigma} g^{\sigma \mu} \bar{e}^{\nu}{}_{\bar{q}} - dB_{\mu \nu} e^{\rho}{}_{p} B_{\rho \sigma} g^{\sigma \mu} \bar{e}^{\nu}{}_{\bar{q}}


결과 요약:

- free index는 $p$, $\bar q$ (맞음)
- $\mu, \nu, \rho, \sigma$ 네 개가 모두 제대로 축약됨
- IndexCalc이 여러 공간(spacetime, lorentz, lorentz̄)을 섞어 써도 bookkeeping을 정확히 함을 확인

## Step 2. Background 값 정의

좌표: $t, r, \theta, \varphi \to 0, 1, 2, 3$.

**FLRW 역계량** (RECALL 식 기반):
$$
\bar g^{\mu\nu} = \operatorname{diag}\!\left(-\tfrac{1}{N^2},\; \tfrac{\Omega^{ij}}{a^2}\right)
$$

**Vielbein** (두 번째 스크린샷, $f \equiv 1 + \tfrac{kr^2}{4}$):
$$
e^{\mu}{}_{p} = \operatorname{diag}\!\left(\tfrac{1}{N},\; \tfrac{f}{a}\,\delta^{i}{}_{p}\right)
$$

DFT 진공 backgroud에선 $\bar e^{\mu}{}_{\bar p}$도 동일한 수치 구조로 가정.

**Background 2-form** $\bar B_{(2)} = \tfrac{h\,r^2\cos\theta}{(1+kr^2/4)^3}\, dr\wedge d\varphi$, 즉
$$
\bar B_{r\varphi} = -\bar B_{\varphi r} = B_0, \quad B_0 \equiv \tfrac{h\,r^2\cos\theta}{f^3}
$$
나머지 성분은 0.

In [2]:
import sympy as sp
from sympy import zeros, simplify, MatrixSymbol, Matrix

N, a, f, B0 = sp.symbols('N a f B0', positive=True)
Om = MatrixSymbol('Ω', 3, 3)   # Ω^{ij}: 3차원 conformal 역계량

# 역계량
g_inv = zeros(4, 4)
g_inv[0, 0] = -1 / N**2
for i in range(3):
    for j in range(3):
        g_inv[1+i, 1+j] = Om[i, j] / a**2

# Vielbein e^μ_p
e = zeros(4, 4)
e[0, 0] = 1 / N
for i in range(3):
    e[1+i, 1+i] = f / a

# 바 씌운 vielbein: 동일 구조 가정
ebar = e

# Background B
B = zeros(4, 4)
B[1, 3] =  B0
B[3, 1] = -B0

print('ḡ^{μν} =')
display(g_inv)
print('\ne^μ_p =')
display(e)
print('\nB̄_{ρσ} =')
display(B)

ḡ^{μν} =


Matrix([
[-1/N**2,            0,            0,            0],
[      0, Ω[0, 0]/a**2, Ω[0, 1]/a**2, Ω[0, 2]/a**2],
[      0, Ω[1, 0]/a**2, Ω[1, 1]/a**2, Ω[1, 2]/a**2],
[      0, Ω[2, 0]/a**2, Ω[2, 1]/a**2, Ω[2, 2]/a**2]])


e^μ_p =


Matrix([
[1/N,   0,   0,   0],
[  0, f/a,   0,   0],
[  0,   0, f/a,   0],
[  0,   0,   0, f/a]])


B̄_{ρσ} =


Matrix([
[0,   0, 0,  0],
[0,   0, 0, B0],
[0,   0, 0,  0],
[0, -B0, 0,  0]])

## Step 3. 중간 텐서 $Y^\mu{}_p$

$$
Y^{\mu}{}_{p} \;=\; \sum_{\rho,\sigma} e^{\rho}{}_{p}\, \bar B_{\rho\sigma}\, \bar g^{\sigma\mu}
$$

**물리적 예측**:
- $\bar B$는 $r\varphi$ 평면에만 있으므로 $\rho \in \{1, 3\}$만 기여
- vielbein은 대각 블록이라 $e^\rho{}_p$가 $\rho=p$일 때만 nonzero
- 결국 $p \in \{1, 3\}$ (즉 $r$, $\varphi$) 일 때만 $Y^\mu{}_p \ne 0$

In [3]:
Y = zeros(4, 4)   # Y[μ, p]
for mu in range(4):
    for p in range(4):
        s = sp.S.Zero
        for rho in range(4):
            for sig in range(4):
                s += e[rho, p] * B[rho, sig] * g_inv[sig, mu]
        Y[mu, p] = simplify(s)

print('Y^μ_p (nonzero 성분):')
for mu in range(4):
    for p in range(4):
        if Y[mu, p] != 0:
            print(f'  Y[μ={mu}, p={p}] = {Y[mu, p]}')

Y^μ_p (nonzero 성분):
  Y[μ=1, p=1] = B0*f*Ω[2, 0]/a**3
  Y[μ=1, p=3] = -B0*f*Ω[0, 0]/a**3
  Y[μ=2, p=1] = B0*f*Ω[2, 1]/a**3
  Y[μ=2, p=3] = -B0*f*Ω[0, 1]/a**3
  Y[μ=3, p=1] = B0*f*Ω[2, 2]/a**3
  Y[μ=3, p=3] = -B0*f*Ω[0, 2]/a**3


예측대로 $p \in \{1, 3\}$에서만 nonzero. $Y^\mu{}_1$은 Ω의 φ-행(인덱스 2)을 뽑아내고, $Y^\mu{}_3$은 r-행(인덱스 0)을 뽑아낸다 (B의 반대칭성 때문에 부호 반전).

## Step 4. $Z^{\mu\nu}{}_{p\bar q} = Y^\mu{}_p \cdot \bar e^\nu{}_{\bar q}$

이건 단순 outer product — 추가 축약 없음. 이어서 $\delta g$, $\delta B$와 $(\mu,\nu)$에 대해 축약해야 하므로 텐서 자체를 저장하지 않고 다음 단계에서 바로 합친다.

## Step 5. 최종 $T_{p\bar q}$ 축약

$$
T_{p\bar q} \;=\; -\sum_{\mu,\nu} \bigl(\delta g_{\mu\nu} + \delta B_{\mu\nu}\bigr)\; Y^{\mu}{}_{p}\, \bar e^{\nu}{}_{\bar q}
$$

$\delta g$와 $\delta B$는 MatrixSymbol로 둬서 결과에 그 성분들이 symbolic하게 남도록 한다.

In [4]:
dg = MatrixSymbol('δg', 4, 4)
dB = MatrixSymbol('δB', 4, 4)

def contract(delta_mat):
    T = zeros(4, 4)
    for p in range(4):
        for qb in range(4):
            s = sp.S.Zero
            for mu in range(4):
                for nu in range(4):
                    s += delta_mat[mu, nu] * Y[mu, p] * ebar[nu, qb]
            T[p, qb] = simplify(-s)
    return T

T_g = contract(dg)
T_B = contract(dB)
T_total = sp.Matrix(4, 4, lambda p, q: simplify(T_g[p, q] + T_B[p, q]))

In [5]:
print('δg 기여분 T^{δg}_{p q̄} (nonzero):')
for p in range(4):
    for qb in range(4):
        if T_g[p, qb] != 0:
            print(f'  [p={p}, q̄={qb}]  {T_g[p, qb]}')

δg 기여분 T^{δg}_{p q̄} (nonzero):
  [p=1, q̄=0]  B0*f*(-Ω[2, 0]*δg[1, 0] - Ω[2, 1]*δg[2, 0] - Ω[2, 2]*δg[3, 0])/(N*a**3)
  [p=1, q̄=1]  B0*f**2*(-Ω[2, 0]*δg[1, 1] - Ω[2, 1]*δg[2, 1] - Ω[2, 2]*δg[3, 1])/a**4
  [p=1, q̄=2]  B0*f**2*(-Ω[2, 0]*δg[1, 2] - Ω[2, 1]*δg[2, 2] - Ω[2, 2]*δg[3, 2])/a**4
  [p=1, q̄=3]  B0*f**2*(-Ω[2, 0]*δg[1, 3] - Ω[2, 1]*δg[2, 3] - Ω[2, 2]*δg[3, 3])/a**4
  [p=3, q̄=0]  B0*f*(Ω[0, 0]*δg[1, 0] + Ω[0, 1]*δg[2, 0] + Ω[0, 2]*δg[3, 0])/(N*a**3)
  [p=3, q̄=1]  B0*f**2*(Ω[0, 0]*δg[1, 1] + Ω[0, 1]*δg[2, 1] + Ω[0, 2]*δg[3, 1])/a**4
  [p=3, q̄=2]  B0*f**2*(Ω[0, 0]*δg[1, 2] + Ω[0, 1]*δg[2, 2] + Ω[0, 2]*δg[3, 2])/a**4
  [p=3, q̄=3]  B0*f**2*(Ω[0, 0]*δg[1, 3] + Ω[0, 1]*δg[2, 3] + Ω[0, 2]*δg[3, 3])/a**4


In [6]:
print('δB 기여분 T^{δB}_{p q̄} (nonzero):')
for p in range(4):
    for qb in range(4):
        if T_B[p, qb] != 0:
            print(f'  [p={p}, q̄={qb}]  {T_B[p, qb]}')

δB 기여분 T^{δB}_{p q̄} (nonzero):
  [p=1, q̄=0]  B0*f*(-Ω[2, 0]*δB[1, 0] - Ω[2, 1]*δB[2, 0] - Ω[2, 2]*δB[3, 0])/(N*a**3)
  [p=1, q̄=1]  B0*f**2*(-Ω[2, 0]*δB[1, 1] - Ω[2, 1]*δB[2, 1] - Ω[2, 2]*δB[3, 1])/a**4
  [p=1, q̄=2]  B0*f**2*(-Ω[2, 0]*δB[1, 2] - Ω[2, 1]*δB[2, 2] - Ω[2, 2]*δB[3, 2])/a**4
  [p=1, q̄=3]  B0*f**2*(-Ω[2, 0]*δB[1, 3] - Ω[2, 1]*δB[2, 3] - Ω[2, 2]*δB[3, 3])/a**4
  [p=3, q̄=0]  B0*f*(Ω[0, 0]*δB[1, 0] + Ω[0, 1]*δB[2, 0] + Ω[0, 2]*δB[3, 0])/(N*a**3)
  [p=3, q̄=1]  B0*f**2*(Ω[0, 0]*δB[1, 1] + Ω[0, 1]*δB[2, 1] + Ω[0, 2]*δB[3, 1])/a**4
  [p=3, q̄=2]  B0*f**2*(Ω[0, 0]*δB[1, 2] + Ω[0, 1]*δB[2, 2] + Ω[0, 2]*δB[3, 2])/a**4
  [p=3, q̄=3]  B0*f**2*(Ω[0, 0]*δB[1, 3] + Ω[0, 1]*δB[2, 3] + Ω[0, 2]*δB[3, 3])/a**4


In [7]:
print('총합 T_{p q̄} = T^{δg} + T^{δB} (nonzero):')
for p in range(4):
    for qb in range(4):
        val = T_total[p, qb]
        if val != 0:
            print(f'  [p={p}, q̄={qb}]  {val}')

총합 T_{p q̄} = T^{δg} + T^{δB} (nonzero):
  [p=1, q̄=0]  B0*f*(-Ω[2, 0]*δB[1, 0] - Ω[2, 0]*δg[1, 0] - Ω[2, 1]*δB[2, 0] - Ω[2, 1]*δg[2, 0] - Ω[2, 2]*δB[3, 0] - Ω[2, 2]*δg[3, 0])/(N*a**3)
  [p=1, q̄=1]  B0*f**2*(-Ω[2, 0]*δB[1, 1] - Ω[2, 0]*δg[1, 1] - Ω[2, 1]*δB[2, 1] - Ω[2, 1]*δg[2, 1] - Ω[2, 2]*δB[3, 1] - Ω[2, 2]*δg[3, 1])/a**4
  [p=1, q̄=2]  B0*f**2*(-Ω[2, 0]*δB[1, 2] - Ω[2, 0]*δg[1, 2] - Ω[2, 1]*δB[2, 2] - Ω[2, 1]*δg[2, 2] - Ω[2, 2]*δB[3, 2] - Ω[2, 2]*δg[3, 2])/a**4
  [p=1, q̄=3]  B0*f**2*(-Ω[2, 0]*δB[1, 3] - Ω[2, 0]*δg[1, 3] - Ω[2, 1]*δB[2, 3] - Ω[2, 1]*δg[2, 3] - Ω[2, 2]*δB[3, 3] - Ω[2, 2]*δg[3, 3])/a**4
  [p=3, q̄=0]  B0*f*(Ω[0, 0]*δB[1, 0] + Ω[0, 0]*δg[1, 0] + Ω[0, 1]*δB[2, 0] + Ω[0, 1]*δg[2, 0] + Ω[0, 2]*δB[3, 0] + Ω[0, 2]*δg[3, 0])/(N*a**3)
  [p=3, q̄=1]  B0*f**2*(Ω[0, 0]*δB[1, 1] + Ω[0, 0]*δg[1, 1] + Ω[0, 1]*δB[2, 1] + Ω[0, 1]*δg[2, 1] + Ω[0, 2]*δB[3, 1] + Ω[0, 2]*δg[3, 1])/a**4
  [p=3, q̄=2]  B0*f**2*(Ω[0, 0]*δB[1, 2] + Ω[0, 0]*δg[1, 2] + Ω[0, 1]*δB[2, 2] + Ω[0, 1]*δg[2, 2] + Ω

## 결과 해석

**Nonzero 성분**은 오직 $p \in \{r, \varphi\}$에서만 나타난다. $p = t$ 또는 $p = \theta$이면 $Y^\mu{}_p = 0$이라 자명하게 0.

패턴을 정리하면:

| $(p, \bar q)$ | 값 (대표 구조) |
|---|---|
| $(r, \bar 0)$   | $-\dfrac{B_0\, f}{N\,a^3}\; \Omega^{\varphi j}\bigl(\delta g_{j0} + \delta B_{j0}\bigr)$ |
| $(r, \bar k)$   | $-\dfrac{B_0\, f^2}{a^4}\; \Omega^{\varphi j}\bigl(\delta g_{jk} + \delta B_{jk}\bigr)$ |
| $(\varphi, \bar 0)$ | $+\dfrac{B_0\, f}{N\,a^3}\; \Omega^{r j}\bigl(\delta g_{j0} + \delta B_{j0}\bigr)$ |
| $(\varphi, \bar k)$ | $+\dfrac{B_0\, f^2}{a^4}\; \Omega^{r j}\bigl(\delta g_{jk} + \delta B_{jk}\bigr)$ |

즉 $\bar B$가 $dr\wedge d\varphi$ 한 성분만 갖는 사실이 그대로 전파되어, 외부 Lorentz 인덱스 $p$가 $r$ 또는 $\varphi$에서만 살아남고 $\Omega^{ij}$의 해당 행을 뽑아낸다.

## IndexCalc 진단

**잘 된 것**
- Multi-space index bookkeeping (spacetime × lorentz × lorentz̄)
- `\bar{e}`, `\bar{q}` decorator 파싱과 LaTeX roundtrip
- Einstein convention 검증 (4중 축약 μνρσ 전부 정확)

**아직 안 되는 것**
1. Component matrix 주입 — 현재 `evaluate/component.py`는 JAX 수치용이라 symbolic 혼합 불가
2. Vielbein identity 자동 적용: $e\cdot\eta\cdot e = g$, $e\cdot e^{-1}=\delta$ 규칙이 없어 $e^\rho{}_p\,\bar B_{\rho\sigma}\,\bar e^\sigma{}_{\bar q}$ 같은 pattern을 Lorentz 성분 $\bar B_{p\bar q}$로 자동 축약 못 함
3. 배경+섭동 혼합 평가 파이프라인 (배경은 대입, 섭동은 심볼 유지) 부재

다음 개발 항목으로 **vielbein identity 자동 인식** 또는 **component substitution API**가 유력.